# 50 Functional Gene Neighbours Seeded  from each Candidate Genes Identified Through Literature Review 

- Gene network is generated for each of the 8 candidate genes from literature review. Homo sapiens genes only. microRNA not included. 

In [7]:
import requests
import json 
import pandas as pd
import os
#import matplotlib.pyplot as plt
#import numpy as np
#from Bio import Entrez
#import sys
#import seaborn as sns
#import math

### Remove redundancy for candidate genes from literature review 

In [8]:
# Remove Candidate Genes that appear in MTX_PK_PD gene sets

with open(os.path.join("data","pk_pd_list.json"), "r") as mtx:
    pk_pd = json.load(mtx)
with open(os.path.join("data","candidate_genes.txt"),"r") as f:
    CANDIDATE = f.read()
CANDIDATE = CANDIDATE.split("\n")
CANDIDATE = [x for x in set(CANDIDATE)if not x in pk_pd]

with open("./gene_set_results/sonis_dict.json") as sonis:
    sonis_dict = json.load(sonis)
    
#Remove Candidate Genes that appear in Sonis gene sets
sonis_genes = [x for y in sonis_dict.values() for x in y] 

CANDIDATE = [x for x in CANDIDATE if not x in list(sonis_genes)] 

CANDIDATE



['XRCC1', 'GHRL', 'ERCC5', 'DPYD', 'ERCC1', 'XRCC6', 'ZNF24', 'ERCC2', 'CAT']

In [9]:
# DPYD is also removed since it is specific for Fluoropyrimidine pathways
CANDIDATE = [
 'ZNF24',
 'ERCC5',
 'GHRL',
 'CAT',
 'XRCC1',
 'XRCC6',
 'ERCC1',
 'ERCC2']

### 50 Functional Gene Neightbours for each of the above Candidate Genes

In [10]:
def find_neighbours(gene, nodes): 

    string_api_url = "https://version-11-5.string-db.org/api"
    output_format = "tsv-no-header"
    method = "interaction_partners"

    ##
    ## Construct the request
    ##

    request_url = "/".join([string_api_url, output_format, method])

    ##
    ## Set parameters
    ##
    my_genes = [gene]
    
    params = {

        "identifiers" : "%0d".join(my_genes), # your protein
        "species" : 9606, # species NCBI identifier 
        "limit" : nodes ,
        "network type" : "functional", 

    }

    ##
    ## Call STRING
    ##

    response = requests.post(request_url, data=params)

    ##
    ## Read and parse the results
    ##
    list = [gene]
    for line in response.text.strip().split("\n"):

        l = line.strip().split("\t")
        query_ensp = l[0]
        query_name = l[2]
        partner_ensp = l[1]
        partner_name = l[3] 
        combined_score = l[5]
        list.append(partner_name)

    return(list)


In [11]:
string_candidate_gene_dict={}
for gene in CANDIDATE:
    string_candidate_gene_dict[gene] = find_neighbours(gene,50)

In [12]:
with open ("./gene_set_results/string_candidate_gene_dict.json", "w") as f:
    json.dump(string_candidate_gene_dict, f)

In [13]:
df = pd.DataFrame({"candidate":list(string_candidate_gene_dict),"gene_set":list(string_candidate_gene_dict.values())})
df

,candidate,gene_set
0,ZNF24,"[ZNF24, ZNF396, ZNF444, ZNF446, SCAND1, ZSCAN3..."
1,ERCC5,"[ERCC5, ERCC4, ERCC1, ERCC3, GTF2H1, ERCC2, ER..."
2,GHRL,"[GHRL, GHSR, MBOAT4, LEP, GPR39, INS, GHRH, IG..."
3,CAT,"[CAT, PEX5, SOD2, SOD1, SCP2, ACOX1, SOD3, FOX..."
4,XRCC1,"[XRCC1, POLB, PNKP, APTX, PARP1, LIG3, APEX1, ..."
5,XRCC6,"[XRCC6, XRCC4, WRN, MSH6, XRCC5, APLF, PRKDC, ..."
6,ERCC1,"[ERCC1, SLX4, XPA, ERCC4, MSH2, MUS81, RAD52, ..."
7,ERCC2,"[ERCC2, CCNH, GTF2H5, GTF2H1, GTF2H3, GTF2H4, ..."
